# 5. Conclusion

## 5.1 Key results

The numbers below are read from the crawl's logs when the book is built; chapters
[3](statistics.ipynb) and [4](performance.ipynb) derive and discuss them.

In [ ]:
import pandas as pd

import report_data as rd

rd.check()
bench, check, fields = rd.benchmark(), rd.corpus_check(), rd.reference_fields()
versions, events = rd.versions(), rd.events()
total_refs = fields["references_total"]
rows = [
    ("Papers scraped (every version processed)",
     f"{bench['papers_scraped']:,} of {bench['ids_assigned']:,} ({bench['success_rate_of_assigned_pct']:.2f}%)"),
    ("Source versions processed", f"{bench['versions_processed']:,} of {bench['versions_total']:,}"),
    ("Average paper size, before → after figure removal",
     f"{bench['avg_paper_bytes_before'] / rd.MB:.2f} MB → {bench['avg_paper_bytes_after'] / rd.MB:.3f} MB "
     f"(−{bench['space_saved_pct']:.2f}%)"),
    ("Reference metadata success rate", f"{bench['reference_success_rate_pct']:.2f}%"),
    ("References per paper (with an arXiv ID)",
     f"{bench['avg_references_per_paper']:.2f} ({bench['avg_arxiv_references_per_paper']:.2f})"),
    ("Entry discovery time", f"{bench['discovery_seconds'] / 60:.1f} min"),
    ("Total wall time / per paper",
     f"{bench['wall_seconds_total'] / 3600:.1f} h / {bench['avg_seconds_per_paper_end_to_end']:.2f} s"),
    ("Median time per source version", f"{versions['seconds'].median():.2f} s"),
    ("RAM, peak / average", f"{bench['ram_peak_mb']:,.0f} MB / {bench['ram_avg_mb']:,.0f} MB"),
    ("Disk, maximum / final output",
     f"{bench['max_disk_bytes'] / rd.GB:.2f} GB / {bench['final_output_bytes'] / rd.GB:.2f} GB"),
]
rd.table(pd.DataFrame(rows, columns=["Result", "Value"]), "Table 5.1. Milestone 1 at a glance.")

## 5.2 What worked

- **Batching turned two of the three stages into minutes.** Asking the arXiv API for 100 IDs per call and
  Semantic Scholar for 100 papers per call removed discovery and references from the running time. The only
  stage that must send one request per item, the source download, is the only one that takes hours.
- **The download runs close to what the request interval allows for small archives.** Unpacking, filtering
  and the writes to Google Drive happen in worker threads during the wait for the next request slot, so a
  small version costs the 3 s interval and nothing more; what remains above the floor is the transfer of
  the large archives.
- **Keeping only `.tex` and `.bib` at extraction time makes the corpus small.** The archives are never
  stored, memory stays flat because they are streamed to the local disk, and the final corpus is a few
  percent of the volume downloaded.
- **Resumability was the feature the crawl depended on most.** It ran over 30 Colab sessions, a good part of
  them interrupted. Per-batch checkpoints, per-version receipts and atomic version folders meant that an
  interruption never cost more than the version in progress.
- **The logs make the result auditable.** Every failed or refused request is recorded with a stable label,
  every version has a receipt with sizes, time and hashes, and every session records its time and memory,
  which is what allows this report to be computed rather than written by hand.

## 5.3 Limitations

In [ ]:
missing = bench["versions_total"] - bench["versions_processed"]
nf_versions = events[events["error_kind"] == "NOT_FOUND"][["id", "version"]].drop_duplicates()
n_404 = len(nf_versions)
n_other = missing - n_404
n_s2_404 = fields["paper_status"].get("not_found", 0)
no_s2 = total_refs - fields["fields_present"]["ref_s2_id"]
rd.md(f'''
- **{missing:,} versions have no Milestone 1 receipt** ({bench['ids_assigned'] - bench['papers_scraped']:,} papers
  are incomplete): **{n_404:,}** versions returned persistent HTTP 404 from `export.arxiv.org/src/` across
  repeated sessions and **{n_other:,}** archives repeatedly failed extraction/safety checks (section 3.1). Following
  the instructor's guidance, all {missing:,} are marked as not downloaded and retained as empty records in
  Milestone 2 (`not_found` and `not_downloaded`).
- **Files of {check['ok_receipts_without_folder']:,} processed versions were missing on Google Drive** when the
  stored corpus was checked on {check['checked_at'][:10]} ({check['papers_affected']:,} papers,
  `{check['first_id']}`–`{check['last_id']}`), although their receipts were synced. `src/09_repair_missing_sources.ipynb`
  re-downloaded all of them before Milestone 2 ran (0 `files_missing` in Milestone 2); the lesson for downstream
  stages is to verify folders after any session that did not end cleanly.
- **The recorded wall time is a lower bound.** A session killed by Colab cannot write its final record, so
  its time is cut at the last periodic save; section 4.2 gives an independent estimate from the receipts.
- **The benchmark is the sum of 30 sessions**, not one uninterrupted run, and Colab runtimes differ in
  network speed and in how their shared IP is treated by arXiv. All waits and retries are included in the
  total runtime.
- **Figure removal is by file extension.** Everything that is not `.tex` or `.bib` is discarded, including
  `.bbl` files. A paper whose bibliography exists only as a compiled `.bbl`, or inline in a `.tex` file,
  has no `.bib` in the corpus (2,231 `missing_bbl` versions in Milestone 2).
- **References depend on Semantic Scholar's coverage.** They are stored once per publication (Semantic Scholar
  does not distinguish versions); **{n_s2_404:,}** papers remained `not_found` (empty `references.json` written);
  and {no_s2 / total_refs:.1%} of the saved references have no Semantic Scholar ID, so they can only be matched
  by title.
''')